# 3D result Export

## Plot Field

In [3]:
import ansys.aedt.core
import os
import tempfile
import time
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.
from ansys.aedt.core import Desktop

import ansys.aedt.core
from ansys.aedt.core import Desktop
import subprocess
import psutil
import time
import os
import re

def get_aedt_processes_detailed():
    """
    실행 중인 AEDT 프로세스를 상세히 확인합니다.
    
    Returns:
    --------
    list : AEDT 프로세스 정보 리스트
    """
    print("🔍 실행 중인 AEDT 프로세스 검색...")
    
    aedt_processes = []
    try:
        for proc in psutil.process_iter(['pid', 'name', 'cmdline', 'create_time']):
            try:
                pinfo = proc.info
                process_name = pinfo['name'] if pinfo['name'] else ""
                
                # AEDT 관련 프로세스 필터링
                if any(keyword in process_name.lower() for keyword in ['ansysedt', 'aedt']):
                    # 포트 정보 추출 시도
                    ports = []
                    try:
                        connections = proc.connections()
                        for conn in connections:
                            if conn.status == 'LISTEN':
                                ports.append(conn.laddr.port)
                    except (psutil.AccessDenied, psutil.NoSuchProcess):
                        pass
                    
                    aedt_processes.append({
                        'pid': pinfo['pid'],
                        'name': process_name,
                        'cmdline': pinfo['cmdline'] if pinfo['cmdline'] else [],
                        'create_time': time.ctime(pinfo['create_time']),
                        'ports': ports
                    })
                    
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
    
        if aedt_processes:
            print(f"✅ {len(aedt_processes)}개의 AEDT 프로세스 발견:")
            for i, proc in enumerate(aedt_processes):
                print(f"\n📋 프로세스 {i+1}:")
                print(f"   PID: {proc['pid']}")
                print(f"   이름: {proc['name']}")
                print(f"   생성시간: {proc['create_time']}")
                if proc['ports']:
                    print(f"   열린 포트: {proc['ports']}")
                else:
                    print(f"   열린 포트: 없음")
        else:
            print("❌ AEDT 프로세스가 없습니다.")
            
        return aedt_processes
        
    except Exception as e:
        print(f"❌ 프로세스 검색 중 오류: {e}")
        return []

def try_connect_to_existing_desktop():
    """
    기존 AEDT Desktop에 연결을 시도합니다.
    
    Returns:
    --------
    Desktop or None : 연결된 Desktop 객체 또는 None
    """
    print("🔗 기존 AEDT Desktop 연결 시도...")
    
    try:
        # 방법 1: new_desktop_session=False로 기존 세션에 연결
        desktop = Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=False,
            non_graphical=NG_MODE
        )
        print("✅ 기존 AEDT Desktop에 성공적으로 연결되었습니다!")
        return desktop
        
    except Exception as e:
        print(f"❌ 기존 Desktop 연결 실패: {e}")
        return None

def try_connect_with_ports(port_list):
    """
    특정 포트들을 시도해서 AEDT에 연결합니다.
    
    Parameters:
    -----------
    port_list : list
        시도할 포트 번호 리스트
        
    Returns:
    --------
    Desktop or None : 연결된 Desktop 객체 또는 None
    """
    AEDT_VERSION='251'
    NG_MODE=False
    for port in port_list:
        try:
            print(f"🔗 포트 {port}로 연결 시도...")
            desktop = Desktop(
                specified_version=AEDT_VERSION,
                new_desktop_session=False,
                port=port,
                non_graphical=NG_MODE
            )
            print(f"✅ 포트 {port}로 성공적으로 연결되었습니다!")
            return desktop
        except Exception as e:
            print(f"❌ 포트 {port} 연결 실패: {e}")
            continue
    
    return None

def get_desktop_connection():
    """
    다양한 방법으로 AEDT Desktop 연결을 시도합니다.
    
    Returns:
    --------
    Desktop : 연결된 Desktop 객체
    """
    print("=" * 60)
    print("🎯 AEDT Desktop 연결 시도")
    print("=" * 60)
    
    # 1. 기존 Desktop 연결 시도
    desktop = try_connect_to_existing_desktop()
    if desktop:
        return desktop
    
    # 2. 프로세스에서 포트 찾아서 연결 시도
    processes = get_aedt_processes_detailed()
    all_ports = []
    
    for proc in processes:
        all_ports.extend(proc['ports'])
    
    if all_ports:
        desktop = try_connect_with_ports(all_ports)
        if desktop:
            return desktop
    
    # 3. 일반적인 AEDT 포트들 시도
    common_ports = [56800, 56801, 56802, 56803, 56804, 56805]
    print("\n🔍 일반적인 AEDT 포트들 시도...")
    desktop = try_connect_with_ports(common_ports)
    if desktop:
        return desktop
    
    # 4. 새로운 Desktop 세션 생성
    print("\n🆕 새로운 AEDT Desktop 세션을 생성합니다...")
    try:
        desktop = Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=True,
            non_graphical=NG_MODE
        )
        print("✅ 새로운 AEDT Desktop이 생성되었습니다!")
        return desktop
    except Exception as e:
        print(f"❌ 새 Desktop 생성 실패: {e}")
        return None

def check_current_desktop_status(desktop):
    """
    현재 Desktop의 상태를 확인합니다.
    
    Parameters:
    -----------
    desktop : Desktop
        확인할 Desktop 객체
    """
    if not desktop:
        print("❌ Desktop 객체가 없습니다.")
        return
    
    try:
        print("\n" + "=" * 40)
        print("📊 현재 Desktop 상태:")
        print("=" * 40)
        
        # 기본 정보
        print(f"AEDT 버전: {desktop.aedt_version_id}")
        print(f"프로세스 ID: {desktop.aedt_process_id}")
        
        # 프로젝트 정보
        try:
            projects = desktop.project_list()
            print(f"\n📁 열린 프로젝트 ({len(projects)}개):")
            for i, proj_name in enumerate(projects, 1):
                print(f"  {i}. {proj_name}")
            
            # 활성 프로젝트
            active_proj = desktop.active_project()
            if active_proj:
                proj_name = active_proj.GetName()
                print(f"\n🎯 활성 프로젝트: {proj_name}")
                
                # 디자인 목록
                try:
                    design_list = active_proj.GetTopDesignList()
                    print(f"📐 디자인 ({len(design_list)}개):")
                    for i, design in enumerate(design_list, 1):
                        print(f"  {i}. {design}")
                        
                    # 활성 디자인
                    active_design = desktop.active_design()
                    if active_design:
                        print(f"🎯 활성 디자인: {active_design.GetName()}")
                        print(f"   디자인 타입: {active_design.GetDesignType()}")
                except:
                    print("디자인 정보 가져오기 실패")
            else:
                print("🎯 활성 프로젝트: 없음")
                
        except Exception as e:
            print(f"프로젝트 정보 가져오기 실패: {e}")
            
    except Exception as e:
        print(f"❌ Desktop 상태 확인 중 오류: {e}")

def smart_aedt_connector():
    """
    스마트 AEDT 연결 함수 - 사용자 친화적 인터페이스
    
    Returns:
    --------
    Desktop : 연결된 Desktop 객체
    """
    print("🚀 스마트 AEDT 연결기를 시작합니다...")
    
    # Desktop 연결 시도
    desktop = get_desktop_connection()
    
    if desktop:
        # 연결 성공 시 상태 확인
        check_current_desktop_status(desktop)
        
        print("\n" + "=" * 60)
        print("🎉 AEDT Desktop 연결이 완료되었습니다!")
        print("💡 다음과 같이 사용할 수 있습니다:")
        print("=" * 60)
        print("# 프로젝트 열기:")
        print("# project = desktop.open_project(r'C:\\path\\to\\your\\project.aedt')")
        print("#")
        print("# Maxwell 객체 생성:")
        print("# m2d = ansys.aedt.core.Maxwell2d(project=desktop, new_desktop=False)")
        print("# m3d = ansys.aedt.core.Maxwell3d(project=desktop, new_desktop=False)")
        print("=" * 60)
        
        return desktop
    else:
        print("❌ AEDT Desktop 연결에 실패했습니다.")
        print("\n🔍 문제 해결 방법:")
        print("1. Ansys AEDT가 설치되어 있는지 확인")
        print("2. AEDT 라이선스가 사용 가능한지 확인")
        print("3. 수동으로 AEDT를 실행한 후 다시 시도")
        return None

# 간단한 사용 함수들
def quick_connect():
    """빠른 연결 - 기존 세션 우선"""
    return try_connect_to_existing_desktop()

def force_new_session():
    """강제로 새 세션 생성"""
    try:
        return Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=True,
            non_graphical=NG_MODE
        )
    except Exception as e:
        print(f"새 세션 생성 실패: {e}")
        return None


c:\Users\HAN_NDESKTOP\.ansys_python_venvs\pyAEDT_conda\Lib\site-packages\ansys\aedt\core\modeler\schematic.py:40: UserWarning: EMIT API is only available for Python 3.8-3.12.
  warnings.warn("EMIT API is only available for Python 3.8-3.12.")


In [2]:
get_aedt_processes_detailed()
m3d=ansys.aedt.core.Maxwell3d(new_desktop=False,version="2025.2")
designName=m3d.design_list
display(designName)
all_objects = m3d.modeler.object_names
for obj_name in all_objects:
    obj = m3d.modeler[obj_name]
materials=m3d.materials.material_keys
excitations=m3d.excitation_names
boundaryObjs=m3d.boundaries
bName = []
bproperties = []
for boundaryObj in boundaryObjs:
    bName.append(boundaryObj.name)
    bproperties.append(boundaryObj.properties)


🔍 실행 중인 AEDT 프로세스 검색...


C:\Users\HAN_NDESKTOP\AppData\Local\Temp\ipykernel_76600\3234595032.py:40: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  connections = proc.connections()


✅ 1개의 AEDT 프로세스 발견:

📋 프로세스 1:
   PID: 25900
   이름: ansysedt.exe
   생성시간: Fri Oct 31 17:01:53 2025
   열린 포트: [50052, 33987, 50178, 2001, 2002, 50052]
PyAEDT INFO: Python version 3.13.5 | packaged by conda-forge | (main, Jun 16 2025, 08:20:19) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\HAN_ND~1\AppData\Local\Temp\pyaedt_HAN_NDESKTOP_d1b08d8d-82ce-4248-aa3b-e677ca5d8ef7.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: Found active AEDT gRPC session on port 50052.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v252\AnsysEM
PyAEDT INFO: No project is defined. Project Hip_Roll_Yaw_U10_20251021_layer1_ANSYSEM_3D exists and has been read.
PyAEDT INFO: Active Design set to Motor-CAD Hip_Roll_Yaw_U10_

['Motor-CAD Hip_Roll_Yaw_U10_20251021_layer1', 'MechanicalDesign1']

PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec


In [3]:
all_objects

['Rotor_Lamination_Primitive',
 '1Magnet1N1_1_1',
 'Stator_Lamination_Primitive',
 'Axle',
 '1Magnet1S2_1_1',
 '1Magnet1N2_1_1',
 '1Magnet1S3_1_1',
 '1Magnet1N3_1_1',
 '1Magnet1S4_1_1',
 '1Magnet1N4_1_1',
 '1Magnet1S5_1_1',
 '1Magnet1N5_1_1',
 '1Magnet1S6_1_1',
 '1Magnet1N6_1_1',
 '1Magnet1S7_1_1',
 '1Magnet1N7_1_1',
 '1Magnet1S8_1_1',
 '1Magnet1N8_1_1',
 '1Magnet1S9_1_1',
 '1Magnet1N9_1_1',
 '1Magnet1S10_1_1',
 '1Magnet1N10_1_1',
 '1Magnet1S11_1_1',
 'Ph1_P1_C7',
 'Ph1_P1_C8',
 'Ph1_P1_C9',
 'Ph1_P1_C10',
 'Ph1_P1_C11',
 'Ph1_P1_C12',
 'Ph2_P1_C1',
 'Ph2_P1_C2',
 'Ph2_P1_C3',
 'Ph2_P1_C4',
 'Ph2_P1_C5',
 'Ph2_P1_C6',
 'Ph2_P1_C12',
 'Ph3_P1_C4',
 'Ph3_P1_C5',
 'Ph3_P1_C6',
 'Ph3_P1_C7',
 'Ph3_P1_C8',
 'Ph3_P1_C9',
 'Rotating_Band',
 'Whole_Region',
 'Ph1_P1_C7_Terminal',
 'Ph1_P1_C8_Terminal',
 'Ph1_P1_C9_Terminal',
 'Ph1_P1_C10_Terminal',
 'Ph1_P1_C11_Terminal',
 'Ph1_P1_C12_Terminal',
 'Ph2_P1_C1_Terminal',
 'Ph2_P1_C2_Terminal',
 'Ph2_P1_C3_Terminal',
 'Ph2_P1_C4_Terminal',
 'Ph2_P

### Plot

In [4]:
py_vista_plot = m3d.post.plot_field(
    quantity="Mag_B", assignment='Rotor_Lamination_Primitive', plot_cad_objs=True, show=False
)
py_vista_plot.isometric_view = True
py_vista_plot.plot(
    export_image_path=os.path.join("D:", "Mag_B.jpg"), show=True)

PyAEDT INFO: Parsing D:\KDH\artemis\Hip_Roll_Yaw_U10_20251021_layer1_ANSYSEM_3D.aedt.
PyAEDT INFO: File D:\KDH\artemis\Hip_Roll_Yaw_U10_20251021_layer1_ANSYSEM_3D.aedt correctly loaded. Elapsed time: 0m 2sec
PyAEDT INFO: aedt file load time 1.5323162078857422
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 2sec
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Active Design set to Motor-CAD Hip_Roll_Yaw_U10_20251021_layer1


Widget(value='<iframe src="http://localhost:60122/index.html?ui=P_0x1d3684f82f0_0&reconnect=auto" class="pyvis…

True

In [7]:
m3d.release_desktop()

PyAEDT INFO: Desktop has been released and closed.


True

## mesh

In [1]:
from ansys.aedt.core.visualization.plot.pyvista import _parse_aedtplt 
pltPath=r"D:\KDH\gitPyAEDT\pyaedt\tests\system\visualization\example_models\T50\vector_field\SurfaceAcForceDensity.aedtplt"
vertices, faces, scalars, log=_parse_aedtplt(pltPath)
vertices=vertices[0]
faces=faces[0]
import matplotlib.pyplot as plt
import pyvista as pv
mesh = pv.PolyData(vertices, faces)
plotter = pv.Plotter()
plotter.add_mesh(mesh, show_edges=True, color="lightblue")
import tkinter as tk
import vtk

pv.set_jupyter_backend('trame')


plotter.add_mesh(mesh, show_edges=True)
# 노드 선택 활성화
def callback(point):
    print(f"선택한 노드 좌표: {point}")

plotter.enable_point_picking(callback=callback, use_mesh=True)

# 인터랙티브 플롯 실행
plotter.show()


AttributeError: __enter__

# 2D Result Export

In [4]:
# e10 Model
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.

In [6]:
aedt_file=r"D:\KDHe10\e10_mobility_ANSYSEM_2D.aedt"
m2d = ansys.aedt.core.Maxwell2d(
    project=aedt_file,
    version=AEDT_VERSION,
    new_desktop=True,
    non_graphical=NG_MODE,
)

PyAEDT INFO: Parsing D:\KDHe10\e10_mobility_ANSYSEM_2D.aedt.
PyAEDT INFO: Python version 3.13.5 | packaged by conda-forge | (main, Jun 16 2025, 08:20:19) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.22.0.


PyAEDT INFO: Returning found Desktop session with PID 59176!
PyAEDT INFO: File D:\KDHe10\e10_mobility_ANSYSEM_2D.aedt correctly loaded. Elapsed time: 0m 1sec
PyAEDT INFO: Project e10_mobility_ANSYSEM_2D has been opened.
PyAEDT INFO: Active Design set to Motor-CAD e10_mobility
PyAEDT INFO: Active Design set to Motor-CAD e10_mobility
PyAEDT INFO: Aedt Objects correctly read


### get Design Variables


In [ ]:
designName=m2d.design_list
display(designName)
all_objects = m2d.modeler.object_names
for obj_name in all_objects:
    obj = m2d.modeler[obj_name]
materials=m2d.materials.material_keys
excitations=m2d.excitation_names
boundaryObjs=m2d.boundaries
bName = []
bproperties = []
for boundaryObj in boundaryObjs:
    bName.append(boundaryObj.name)
    bproperties.append(boundaryObj.properties)


['Motor-CAD e10_mobility']

PyAEDT INFO: Modeler2D class has been initialized!
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec


### Field Export

In [8]:
setup=m2d.get_setup(name='Setup1')

In [13]:
setup.props["SaveFieldsType"] = "Every N Steps"
setup.props["N Steps"] = "1"

setup.update()


True

In [15]:
m2d.analyze_setup(name=setup.name, cores=NUM_CORES)


PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 2D correctly changed.
PyAEDT INFO: Solving design setup Setup1
PyAEDT INFO: Design setup Setup1 solved correctly in 0.0h 7.0m 22.0s
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 2D correctly changed.


True

In [15]:
ansys.aedt.core.Maxwell2d.omeshmodule

In [17]:
temp_folder=r'D:\KDHe10'

In [ ]:
py_vista_plot = m2d.post.plot_field(
    quantity="Mag_B", assignment='Rotor_1', plot_cad_objs=True, show=False
)
py_vista_plot.isometric_view = True
py_vista_plot.plot(
    export_image_path=os.path.join("D:", "Mag_B.jpg"), show=True)

PyAEDT INFO: Active Design set to Motor-CAD e10_mobility
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec


In [22]:
    mag_j_field_export = m2d.post.export_field_plot(
        plot_name="B",
        output_dir=temp_folder,
        file_format="aedtplt",
    )


PyAEDT ERROR: aedtplt file format is not supported for this plot.


# Backup

In [1]:
# AEDT Maxwell GUI 실행
import pyaedt
print("=== ANSYS Electronics Desktop (AEDT) Maxwell GUI 실행 ===")

# GUI 모드로 AEDT 실행
print("Maxwell 2D GUI를 실행합니다...")

try:
    # GUI 모드로 Maxwell 2D 새 프로젝트 생성
    m2d_gui = pyaedt.Maxwell2d(
        non_graphical=False,  # GUI 모드로 실행
        new_desktop_session=True,  # 새로운 데스크톱 세션 시작
        close_on_exit=False,  # 종료 시 자동으로 닫지 않음
        student_version=False
    )
    
    print(f"✓ Maxwell 2D GUI가 성공적으로 실행되었습니다!")
    print(f"  - 프로젝트명: {m2d_gui.project_name}")
    print(f"  - 디자인명: {m2d_gui.design_name}")
    print(f"  - AEDT 버전: {m2d_gui.aedt_version_id}")
    print(f"  - 솔루션 타입: {m2d_gui.solution_type}")
    
    # GUI 창 정보
    print(f"\n📺 AEDT Maxwell GUI 창이 열렸습니다!")
    print("  - Maxwell 2D TransientXY 환경")
    print("  - 새로운 프로젝트로 시작")
    print("  - 모델링, 해석, 포스트프로세싱 가능")
    
    # 객체를 전역 변수로 저장
    globals()['maxwell_gui'] = m2d_gui
    
except Exception as e:
    print(f"❌ GUI 실행 중 오류 발생: {e}")
    print("다음 사항을 확인해주세요:")
    print("  1. ANSYS Electronics Desktop이 설치되어 있는지")
    print("  2. 라이센스가 유효한지")
    print("  3. 다른 AEDT 세션이 실행 중인지")

print("\n🚀 Maxwell GUI가 준비되었습니다!")

=== ANSYS Electronics Desktop (AEDT) Maxwell GUI 실행 ===
Maxwell 2D GUI를 실행합니다...
PyAEDT WARNING: Argument `new_desktop_session` is deprecated for method `__init__`; use `new_desktop` instead.
PyAEDT INFO: Python version 3.13.5 | packaged by conda-forge | (main, Jun 16 2025, 08:20:19) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.17.2.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\HAN_ND~1\AppData\Local\Temp\pyaedt_HAN_NDESKTOP_80ee7931-bc56-4735-a8de-8b2b8eb40c55.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: Python version 3.13.5 | packaged by conda-forge | (main, Jun 16 2025, 08:20:19) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.17.2.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on fil

c:\Users\HAN_NDESKTOP\.ansys_python_venvs\pyAEDT_conda\Lib\site-packages\ansys\aedt\core\modeler\schematic.py:39: UserWarning: EMIT API is only available for Python 3.8-3.12.
  warnings.warn("EMIT API is only available for Python 3.8-3.12.")


PyAEDT INFO: New AEDT session is starting on gRPC port 57218.
PyAEDT INFO: Electronics Desktop started on gRPC port: 57218 after 11.03948426246643 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v251\AnsysEM
PyAEDT INFO: Electronics Desktop started on gRPC port: 57218 after 11.03948426246643 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v251\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.1 version started with process ID 2040.
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.1 version started with process ID 2040.
PyAEDT INFO: Project Project16 has been created.
PyAEDT INFO: Project Project16 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'Maxwell 2D_MN7' of type Maxwell 2D.
PyAEDT INFO: Added design 'Maxwell 2D_MN7' of type Maxwell 2D.
PyAEDT INFO: Aedt Objects correctly read
✓ Maxwell 2D GUI가 성공적으로 실행되었습니다!
  - 프로젝트명: Pro